# Hackathon Setup: Lakebase & Genie Space

This notebook provisions per-user infrastructure for today's hackathon training:
1. A **Lakebase** autoscaling Postgres instance (0.5 to 4 CU)
2. Two **Genie Spaces** (Bakehouse analytics + Detroit 911 incidents)

Each resource is prefixed with the current user's name to avoid collisions. Just run all cells top to bottom!

In [0]:
%pip install -q --upgrade databricks-sdk>=0.118.0
dbutils.library.restartPython()

In [0]:
import json
import re

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Grab current user info
user_info = w.current_user.me()
user_email = user_info.user_name
display_name = user_info.display_name or user_email.split("@")[0]

# RFC 1123 compliant: lowercase, alphanumeric + hyphens, starts with a letter
clean_name = re.sub(r"[^a-z0-9]+", "-", display_name.lower()).strip("-")
if clean_name and not clean_name[0].isalpha():
    clean_name = f"u-{clean_name}"

print(f"User: {display_name} ({user_email})")
print(f"Resource prefix: {clean_name}")

## Step 1: Create Lakebase Autoscaling Instance

Creates a Lakebase Postgres project with autoscaling (0.5 to 4 CU, scale-to-zero enabled).
The project auto-provisions a `production` branch and a `primary` read-write endpoint.

In [0]:
from databricks.sdk.service.postgres import (
    Endpoint,
    EndpointSpec,
    EndpointType,
    FieldMask,
    Project,
    ProjectSpec,
)

project_id = f"{clean_name}-hackathon"

# Create the project (auto-provisions production branch + primary endpoint)
print(f"Creating Lakebase project: {project_id}")
try:
    op = w.postgres.create_project(
        project=Project(spec=ProjectSpec(
            display_name=f"{display_name} Hackathon DB",
            pg_version=17,
        )),
        project_id=project_id,
    )
    project = op.wait()
    print(f"Project created: {project.name}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Project '{project_id}' already exists, continuing...")
    else:
        raise

# Configure autoscaling on the primary endpoint (default is 1 CU fixed)
branch_name = f"projects/{project_id}/branches/production"
endpoints = list(w.postgres.list_endpoints(parent=branch_name))
primary_ep = endpoints[0]

print("Configuring autoscaling: 0.5 to 4 CU")
w.postgres.update_endpoint(
    name=primary_ep.name,
    endpoint=Endpoint(
        name=primary_ep.name,
        spec=EndpointSpec(
            endpoint_type=EndpointType.ENDPOINT_TYPE_READ_WRITE,
            autoscaling_limit_min_cu=0.5,
            autoscaling_limit_max_cu=4.0,
        ),
    ),
    update_mask=FieldMask(field_mask=[
        "spec.autoscaling_limit_min_cu",
        "spec.autoscaling_limit_max_cu",
    ]),
).wait()

# Print the final state
ep = w.postgres.get_endpoint(name=primary_ep.name)
print(f"\nLakebase ready!")
print(f"  Host: {ep.status.hosts.host}")
if ep.spec:
    print(f"  Autoscaling: {ep.spec.autoscaling_limit_min_cu} - {ep.spec.autoscaling_limit_max_cu} CU")
else:
    print(f"  Autoscaling: 0.5 - 4.0 CU (configured)")
print(f"  State: {ep.status.current_state}")

## Step 2: Create Genie Spaces

Creates two Genie Agent spaces (skips any that already exist):
1. **Bakehouse Analytics** - all 6 tables in `samples.bakehouse`
2. **Detroit 911 Incidents** - `il_sandbox.detroit_911.incidents_silver`

In [0]:
def get_warehouse_id(client: WorkspaceClient) -> str:
    """Find the first available SQL warehouse in the workspace."""
    for wh in client.warehouses.list():
        return wh.id
    raise RuntimeError("No SQL warehouses found in workspace")


def find_existing_space(client: WorkspaceClient, title: str) -> dict | None:
    """Return an existing Genie space whose title starts with the given title.

    Uses startswith because the API appends a timestamp to duplicate titles,
    so an exact match would miss spaces from earlier runs.
    """
    resp = client.genie.list_spaces()
    while True:
        for space in resp.spaces or []:
            if space.title and space.title.startswith(title):
                return {"space_id": space.space_id, "title": space.title}
        if not resp.next_page_token:
            break
        resp = client.genie.list_spaces(page_token=resp.next_page_token)
    return None


def create_genie_space(
    client: WorkspaceClient,
    title: str,
    description: str,
    table_identifiers: list[str],
    parent_path: str,
    warehouse_id: str,
) -> dict:
    """Create a Genie Agent space, or return the existing one if already created.

    Uses w.api_client.do() since w.genie.create_space() is not yet
    available in the current SDK release. Checks for an existing space
    with the same title first to make re-runs safe.

    Args:
        client: Authenticated WorkspaceClient.
        title: Display name for the space.
        description: What this space is about.
        table_identifiers: Fully qualified table names (catalog.schema.table).
        parent_path: Workspace path to create the space under.
        warehouse_id: SQL warehouse ID to back the Genie space.

    Returns:
        Dict with at least 'space_id' for the created or existing space.
    """
    existing = find_existing_space(client, title)
    if existing:
        print(f"  Space '{title}' already exists, skipping creation.")
        return existing

    tables = [{"identifier": t} for t in table_identifiers]

    serialized_space = json.dumps({
        "version": 1,
        "config": {},
        "data_sources": {"tables": tables},
        "instructions": {},
    })

    payload = {
        "title": title,
        "description": description,
        "parent_path": parent_path,
        "warehouse_id": warehouse_id,
        "serialized_space": serialized_space,
    }

    return client.api_client.do("POST", "/api/2.0/genie/spaces", body=payload)


parent_path = f"/Workspace/Users/{user_email}"
warehouse_id = get_warehouse_id(w)
print(f"Genie space helper ready (warehouse: {warehouse_id})")

In [0]:
bakehouse_tables = [
    "samples.bakehouse.media_customer_reviews",
    "samples.bakehouse.media_gold_reviews_chunked",
    "samples.bakehouse.sales_customers",
    "samples.bakehouse.sales_franchises",
    "samples.bakehouse.sales_suppliers",
    "samples.bakehouse.sales_transactions",
]

print("Creating Bakehouse Genie Space...")
bakehouse_space = create_genie_space(
    client=w,
    title=f"{clean_name} - Bakehouse Analytics",
    description="Explore bakehouse sales, franchises, suppliers, and customer reviews.",
    table_identifiers=bakehouse_tables,
    parent_path=parent_path,
    warehouse_id=warehouse_id,
)
bakehouse_id = bakehouse_space.get("space_id", "N/A")
print(f"Bakehouse space created! ID: {bakehouse_id}")

In [0]:
detroit_tables = ["il_sandbox.detroit_911.incidents_silver"]

print("Creating Detroit 911 Genie Space...")
detroit_space = create_genie_space(
    client=w,
    title=f"{clean_name} - Detroit 911 Incidents",
    description="Analyze Detroit 911 incident data: call types, response times, and geographic patterns.",
    table_identifiers=detroit_tables,
    parent_path=parent_path,
    warehouse_id=warehouse_id,
)
detroit_id = detroit_space.get("space_id", "N/A")
print(f"Detroit 911 space created! ID: {detroit_id}")

In [0]:
print("=" * 55)
print("HACKATHON SETUP COMPLETE!")
print("=" * 55)
print(f"\nUser: {display_name}")
print(f"\nLakebase:")
print(f"  Project: {project_id}")
print(f"  Host: {ep.status.hosts.host}")
print(f"\nGenie Spaces:")
print(f"  1. {clean_name} - Bakehouse Analytics (ID: {bakehouse_id})")
print(f"  2. {clean_name} - Detroit 911 Incidents (ID: {detroit_id})")
print(f"\nYou're ready to hack!")